# Advanced Pandas
## GroupBy, merging, pivot tables, and performance optimization

## 1. Sample Data Setup

In [ ]:
import pandas as pd
import numpy as np

# Create sample sales data
sales_data = {
    'Date': pd.date_range('2023-01-01', periods=12, freq='MS'),
    'Department': ['Sales', 'IT', 'HR', 'Sales', 'IT', 'HR'] * 2,
    'Product': ['A', 'B', 'C', 'A', 'B', 'C'] * 2,
    'Amount': [100, 200, 150, 120, 210, 160, 110, 220, 155, 130, 215, 165],
    'Quantity': [10, 20, 15, 12, 21, 16, 11, 22, 15, 13, 21, 16]
}

df_sales = pd.DataFrame(sales_data)
print("Sales Data:")
print(df_sales.head())

## 2. GroupBy Operations

In [ ]:
# GroupBy with single column
grouped_dept = df_sales.groupby('Department')['Amount'].sum()
print("Total amount by department:")
print(grouped_dept)

# GroupBy with multiple columns
grouped_multi = df_sales.groupby(['Department', 'Product'])['Amount'].sum()
print("\nTotal amount by department and product:")
print(grouped_multi)

In [ ]:
# Multiple aggregations
agg_results = df_sales.groupby('Department')['Amount'].agg(['sum', 'mean', 'count', 'std'])
print("Multiple aggregations:")
print(agg_results)

# Named aggregations
agg_named = df_sales.groupby('Department').agg(
    total_amount=('Amount', 'sum'),
    avg_amount=('Amount', 'mean'),
    num_records=('Amount', 'count')
)
print("\nNamed aggregations:")
print(agg_named)

In [ ]:
# Transform - broadcast result back to original shape
df_sales['dept_total'] = df_sales.groupby('Department')['Amount'].transform('sum')
print("GroupBy transform:")
print(df_sales[['Department', 'Amount', 'dept_total']].head())

# Calculate percentage of total
df_sales['pct_of_dept'] = df_sales['Amount'] / df_sales['dept_total']
print("\nPercentage of department total:")
print(df_sales[['Department', 'Amount', 'pct_of_dept']].head())

## 3. Merging and Joining

In [ ]:
# Create sample datasets
df_employees = pd.DataFrame({
    'EmployeeID': [1, 2, 3, 4],
    'Name': ['Alice', 'Bob', 'Charlie', 'David'],
    'Department': ['Sales', 'IT', 'HR', 'Sales']
})

df_salaries = pd.DataFrame({
    'EmployeeID': [1, 2, 3, 5],
    'Salary': [50000, 60000, 55000, 52000]
})

print("Employees:")
print(df_employees)
print("\nSalaries:")
print(df_salaries)

In [ ]:
# Inner merge
inner_merge = pd.merge(df_employees, df_salaries, on='EmployeeID', how='inner')
print("Inner merge:")
print(inner_merge)

# Left merge
left_merge = pd.merge(df_employees, df_salaries, on='EmployeeID', how='left')
print("\nLeft merge:")
print(left_merge)

# Right merge
right_merge = pd.merge(df_employees, df_salaries, on='EmployeeID', how='right')
print("\nRight merge:")
print(right_merge)

# Outer merge
outer_merge = pd.merge(df_employees, df_salaries, on='EmployeeID', how='outer')
print("\nOuter merge:")
print(outer_merge)

## 4. Pivot Tables

In [ ]:
# Create pivot table
pivot = df_sales.pivot_table(
    values='Amount',
    index='Department',
    columns='Product',
    aggfunc='sum',
    fill_value=0
)

print("Pivot table (Amount by Department and Product):")
print(pivot)

# Pivot with multiple aggregations
pivot_multi = df_sales.pivot_table(
    values=['Amount', 'Quantity'],
    index='Department',
    columns='Product',
    aggfunc={'Amount': 'sum', 'Quantity': 'mean'}
)

print("\nPivot with multiple aggregations:")
print(pivot_multi)

## 5. Reshaping: Melt, Stack, Unstack

In [ ]:
# Melt - wide to long format
df_wide = pd.DataFrame({
    'Date': ['2023-01', '2023-02'],
    'ProductA': [100, 120],
    'ProductB': [200, 210],
    'ProductC': [150, 160]
})

print("Wide format:")
print(df_wide)

df_long = df_wide.melt(
    id_vars=['Date'],
    value_vars=['ProductA', 'ProductB', 'ProductC'],
    var_name='Product',
    value_name='Amount'
)

print("\nLong format (after melt):")
print(df_long)

In [ ]:
# Stack - convert columns to rows
stacked = pivot.stack()
print("Stacked pivot table:")
print(stacked)

# Unstack - convert rows to columns
unstacked = stacked.unstack()
print("\nUnstacked (back to original):")
print(unstacked)

## 6. Window Functions: Rolling and Expanding

In [ ]:
# Create time series data
df_ts = df_sales.sort_values('Date').reset_index(drop=True)

# Rolling average
df_ts['rolling_avg_3'] = df_ts['Amount'].rolling(window=3).mean()
df_ts['rolling_sum_3'] = df_ts['Amount'].rolling(window=3).sum()

print("Rolling window operations:")
print(df_ts[['Date', 'Amount', 'rolling_avg_3', 'rolling_sum_3']].head(8))

In [ ]:
# Expanding window
df_ts['expanding_sum'] = df_ts['Amount'].expanding().sum()
df_ts['expanding_mean'] = df_ts['Amount'].expanding().mean()

print("Expanding window operations:")
print(df_ts[['Date', 'Amount', 'expanding_sum', 'expanding_mean']].head(8))

## 7. Performance Tips

In [ ]:
# Using categorical dtype for memory efficiency
print("Before categorical:")
print(f"Memory usage: {df_sales.memory_usage(deep=True).sum() / 1024:.2f} KB")

# Convert to categorical
df_cat = df_sales.copy()
df_cat['Department'] = df_cat['Department'].astype('category')
df_cat['Product'] = df_cat['Product'].astype('category')

print("\nAfter categorical:")
print(f"Memory usage: {df_cat.memory_usage(deep=True).sum() / 1024:.2f} KB")
print(f"Memory saved: {(1 - df_cat.memory_usage(deep=True).sum() / df_sales.memory_usage(deep=True).sum()) * 100:.1f}%")

In [ ]:
# Using eval for speed
import time

# Create larger dataset
large_df = pd.DataFrame({
    'A': np.random.rand(100000),
    'B': np.random.rand(100000),
    'C': np.random.rand(100000)
})

# Standard way
start = time.time()
result1 = large_df['A'] + large_df['B'] * large_df['C']
time1 = time.time() - start

# Using eval
start = time.time()
result2 = large_df.eval('A + B * C')
time2 = time.time() - start

print(f"Standard: {time1*1000:.3f} ms")
print(f"Using eval: {time2*1000:.3f} ms")
print(f"Speedup: {time1/time2:.1f}x")